### **BóSight Week 3: Individual Cow Re-Identification**
#
#### Trains a ResNet-50 classifier to identify which of the 16 cows (C01-C16)
#### each detected crop belongs to, based on their unique Holstein-Friesian
#### coat patterns. The workflow was split across two platforms: crop generation
#### on Colab CPU and model training on Kaggle T4 GPU.
#
#### Pipeline:
####   1. Copy images and original labels (with cow_id 1-16) to local storage
####   2. Rebuild the same temporal split from week 2 for consistency
####   3. Crop each labelled bounding box from every frame, save into folders
####      by cow identity (C01/, C02/, ..., C16/)
####   4. Zip crops and upload to Kaggle
####   5. Fine-tune ResNet-50 as a 16-class classifier
####   6. Evaluate on held-out test set
#
#### Approach: classification (softmax) over metric learning (triplet/ArcFace)
#### because this is a closed-set problem, always the same 16 cows.
#
#### **Results:**
####   Val accuracy:  97.09%
####   Test accuracy: 97.0%
####   Macro F1:      0.97
#####  All 16 cows above 93% precision and 95% recall
#
#### Note: Part 1 paths assume Colab with Google Drive mounted at /content/drive.
#### Part 2 paths assume Kaggle, including a dataset path under a specific
#### username; update CROPS_ROOT to your own Kaggle dataset location. Part 2
#### requires the zipped crops from Part 1 to be manually uploaded to Kaggle
#### as a dataset first, the two parts cannot run in one continuous session.

#### Part 1: Crop Geneation
Platform : Colab (GPU)

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Imports and copy data to local storage.
# We need the original labels (with cow_id 1-16) for identity supervision,
# not the remapped class-0 labels from week 2.
import shutil
import random
from pathlib import Path
from collections import defaultdict
from PIL import Image

DRIVE_IMAGES = Path("/content/drive/MyDrive/MmCows/extracted/visual_data/images/0725/cam_1")
DRIVE_LABELS = Path("/content/drive/MyDrive/MmCows/extracted/visual_data/labels/combined/0725/cam_1")
LOCAL_ROOT = Path("/content/bosight_local")

local_images = LOCAL_ROOT / "images"
local_labels = LOCAL_ROOT / "labels"

if not local_images.exists():
    print("Copying images...")
    shutil.copytree(DRIVE_IMAGES, local_images)
if not local_labels.exists():
    print("Copying labels...")
    shutil.copytree(DRIVE_LABELS, local_labels)

# Verify counts and confirm labels have cow IDs (not remapped 0s)
img_count = len(list(local_images.glob("*.jpg")))
lbl_count = len(list(local_labels.glob("*.txt")))
print(f"Images: {img_count}, Labels: {lbl_count}")

sample = next(f for f in local_labels.glob("*.txt") if f.read_text().strip())
print(f"\nSample label ({sample.name}):")
print(sample.read_text()[:200])

In [ ]:
# Rebuild temporal split and generate crops.
# Uses the exact same split logic and seed as before, so the train/val/test
# membership is identical. Crops are saved into folders by cow identity
# so PyTorch's ImageFolder can load them directly.

SEED = 42
random.seed(SEED)
CROP_ROOT = Path("/content/bosight_crops")
WINDOW_SECONDS = 300  # 5-minute windows, same as week 2

def parse_stem(stem):
    """Extract Unix timestamp and hour from filename stem."""
    ts_str, hms = stem.split("_", 1)
    return int(ts_str), int(hms.split("-")[0])

stems = sorted(p.stem for p in local_images.glob("*.jpg"))

# Group frames into 5-minute windows (same logic as week 2)
window_to_stems = defaultdict(list)
window_to_hour = {}
for stem in stems:
    ts, hour = parse_stem(stem)
    wid = ts // WINDOW_SECONDS
    window_to_stems[wid].append(stem)
    window_to_hour[wid] = hour

hour_to_windows = defaultdict(list)
for wid, hour in window_to_hour.items():
    hour_to_windows[hour].append(wid)

# Assign windows to splits (same ratios and seed as week 2)
split_stems = {"train": [], "val": [], "test": []}
for hour, windows in sorted(hour_to_windows.items()):
    windows = windows[:]
    random.shuffle(windows)
    n = len(windows)
    n_train = round(n * 0.70)
    n_val = round(n * 0.15)
    assign = ["train"] * n_train + ["val"] * n_val + ["test"] * (n - n_train - n_val)
    for wid, split in zip(windows, assign):
        split_stems[split].extend(window_to_stems[wid])

# Build stem -> split lookup
stem_to_split = {}
for split, s_list in split_stems.items():
    for s in s_list:
        stem_to_split[s] = split

print(f"Split: train={len(split_stems['train'])}, val={len(split_stems['val'])}, test={len(split_stems['test'])}")

# Crop each labelled bounding box and save by cow identity
# Output structure: bosight_crops/{split}/{cow_name}/{stem}_{bbox_index}.jpg
crop_count = defaultdict(lambda: defaultdict(int))
total = 0

for lbl_path in sorted(local_labels.glob("*.txt")):
    stem = lbl_path.stem
    split = stem_to_split.get(stem)
    if split is None:
        continue

    lines = [l.strip() for l in lbl_path.read_text().strip().split("\n") if l.strip()]
    if not lines:
        continue

    img_path = local_images / f"{stem}.jpg"
    img = Image.open(img_path)
    w, h = img.size  # 4480, 2800

    for i, line in enumerate(lines):
        parts = line.split()
        cow_id = int(parts[0])
        xc, yc, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])

        # Convert normalised YOLO coords to pixel coords
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        # Clamp to image bounds
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)

        # Skip tiny crops (likely annotation noise)
        if x2 - x1 < 10 or y2 - y1 < 10:
            continue

        crop = img.crop((x1, y1, x2, y2))

        # Save as bosight_crops/{split}/C{id}/{stem}_{i}.jpg
        cow_name = f"C{cow_id:02d}"
        out_dir = CROP_ROOT / split / cow_name
        out_dir.mkdir(parents=True, exist_ok=True)
        crop.save(out_dir / f"{stem}_{i}.jpg")

        crop_count[split][cow_name] += 1
        total += 1

# Print crop distribution
# Expected: ~53,508 total, all 16 cows represented in every split
print(f"\nTotal crops: {total}")
for split in ["train", "val", "test"]:
    counts = crop_count[split]
    print(f"\n{split}: {sum(counts.values())} crops")
    for cow in sorted(counts):
        print(f"  {cow}: {counts[cow]}")


In [ ]:
# Zip crops for Kaggle upload.
# !zip -r "/content/drive/MyDrive/MmCows/bosight_crops.zip" /content/bosight_crops

#### Part 2: Model Training
Platform: Kaggle (T4 GPU)


In [ ]:
# Install dependencies
# !pip install timm -q

In [ ]:
# Setup datasets and dataloaders.
# ImageFolder automatically reads C01/, C02/, ..., C16/ as class labels
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms, datasets
import timm
from pathlib import Path
import time

CROPS_ROOT = Path("/kaggle/input/datasets/anandhvenkataraman/crop-cows/content/bosight_crops")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# Training transforms: augmentation to reduce overfitting
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),       # coat pattern is not symmetric
    transforms.RandomRotation(10),           # slight angle variation
    transforms.ColorJitter(brightness=0.2, contrast=0.2),  # lighting variation
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),  # ImageNet stats
])

# Validation/test transforms: no augmentation, just resize and normalise
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Load datasets
train_ds = datasets.ImageFolder(CROPS_ROOT / "train", transform=train_transform)
val_ds = datasets.ImageFolder(CROPS_ROOT / "val", transform=val_transform)
test_ds = datasets.ImageFolder(CROPS_ROOT / "test", transform=val_transform)

# num_workers=4 works on Kaggle; 0 and 2 can cause hangs
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"Classes: {train_ds.classes}")
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

In [ ]:
# Build model and train.
# ResNet-50 pretrained on ImageNet, final layer replaced with 16 outputs.
# Adam optimizer with ReduceLROnPlateau scheduler halves LR when val loss
# plateaus. Best model saved by validation accuracy.

model = timm.create_model("resnet50", pretrained=True, num_classes=16)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

NUM_EPOCHS = 20
best_val_acc = 0.0

for epoch in range(NUM_EPOCHS):
    start = time.time()

    # Training loop
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * imgs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += imgs.size(0)

    # Validation loop
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * imgs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += imgs.size(0)

    train_acc = train_correct / train_total
    val_acc = val_correct / val_total
    avg_val_loss = val_loss / val_total
    scheduler.step(avg_val_loss)
    elapsed = time.time() - start

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | {elapsed:.0f}s | "
          f"Train Loss: {train_loss/train_total:.4f} Acc: {train_acc:.4f} | "
          f"Val Loss: {avg_val_loss:.4f} Acc: {val_acc:.4f}")

    # Save best model by validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "/kaggle/working/reid_best.pt")
        print(f"  Saved best model (val_acc={val_acc:.4f})")

print(f"\nBest val accuracy: {best_val_acc:.4f}")

In [ ]:
# Test set evaluation.
# Load best checkpoint and run classification report on held-out test crops
from sklearn.metrics import classification_report

model.load_state_dict(torch.load("/kaggle/working/reid_best.pt"))
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(1).cpu()
        all_preds.extend(preds.numpy())
        all_labels.extend(labels.numpy())

print(classification_report(all_labels, all_preds, target_names=test_ds.classes))

In [ ]:
# Download weights
from IPython.display import FileLink
display(FileLink("/kaggle/working/reid_best.pt"))